In [1]:
"""
FINAL WORKING TEST CODE FOR ml_classifier.py

"""

from quality_assessment import QualityAssessmentPipeline
from quality_assessment.ml_classifier import QualityClassifier
from pyspark.sql.functions import col, when
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def test_basic_import():
    """Test 1: Can we import the classifier?"""
    print("\n" + "="*80)
    print("TEST 1: IMPORT TEST")
    print("="*80)
    
    try:
        from quality_assessment.ml_classifier import QualityClassifier
        print("✅ SUCCESS: QualityClassifier imported successfully!")
        return True
    except Exception as e:
        print(f"❌ FAILED: {e}")
        return False


def test_initialization():
    """Test 2: Can we initialize the classifier?"""
    print("\n" + "="*80)
    print("TEST 2: INITIALIZATION TEST")
    print("="*80)
    
    try:
        pipeline = QualityAssessmentPipeline()
        spark = pipeline.spark

        # Add these 2 lines to fix Spark timeout
        spark.sparkContext.getConf() \
            .set('spark.executor.heartbeatInterval', '30s') \
            .set('spark.network.timeout', '600s')
        
        classifier = QualityClassifier(spark)
        classifier = QualityClassifier(pipeline.spark)
        print("✅ SUCCESS: QualityClassifier initialized!")
        return True, pipeline, classifier
    except Exception as e:
        print(f"❌ FAILED: {e}")
        return False, None, None


def test_load_data(pipeline, classifier):
    """Test 3: Can we load and prepare data?"""
    print("\n" + "="*80)
    print("TEST 3: DATA LOADING TEST")
    print("="*80)
    
    try:
        # Load Titanic dataset with correct file path
        print("Loading Titanic dataset...")
        titanic_path = 'data/raw/titanic/train.csv'
        df_raw = pipeline.data_loader.load_file(titanic_path)
        print(f"✅ Loaded {df_raw.count()} rows")
        
        # Calculate total_cols from raw DataFrame BEFORE feature engineering
        print("Calculating column count...")
        total_cols = len(df_raw.columns)
        print(f"   Total columns in raw data: {total_cols}")
        
        # Extract features
        print("Extracting features...")
        df_features, _, _ = pipeline.feature_engineer.engineer_all_features(df_raw)
        print(f"✅ Extracted features from {df_features.count()} rows")
        print(f"   Feature columns: {len(df_features.columns)}")
        
        # Create null_ratio using the correct total_cols
        print("Creating null_ratio feature...")
        df_features = df_features.withColumn(
            'null_ratio',
            col('null_count').cast('double') / total_cols
        )
        print(f"✅ Created null_ratio = null_count / {total_cols}")
        
        # Verify null_ratio was created
        print("Verifying null_ratio...")
        sample_data = df_features.select('null_count', 'null_ratio').take(2)
        for i, row in enumerate(sample_data):
            print(f"   Row {i+1}: null_count={row['null_count']}, null_ratio={row['null_ratio']:.4f}")
        
        # Create labels based on null_ratio
        print("Creating ground truth labels...")
        df_labeled = df_features.withColumn(
            'ground_truth',
            when(col('null_ratio') <= 0.1, 1).otherwise(0)
        )
        print(f"✅ Created labels")
        
        # Check label distribution
        label_counts = df_labeled.groupBy('ground_truth').count().collect()
        print(f"   Label distribution:")
        for row in label_counts:
            label = int(row['ground_truth'])
            count = row['count']
            pct = (count / df_labeled.count()) * 100
            print(f"     Label {label}: {count} samples ({pct:.1f}%)")
        
        return True, df_labeled
    except Exception as e:
        print(f"❌ FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False, None


def test_train_random_forest(classifier, df_labeled):
    """Test 4: Can we train the model?"""
    print("\n" + "="*80)
    print("TEST 4: TRAINING TEST")
    print("="*80)
    
    try:
        # Split data
        print("Splitting data (80/20)...")
        train, test = df_labeled.randomSplit([0.8, 0.2], seed=42)
        train_count = train.count()
        test_count = test.count()
        print(f"✅ Train set: {train_count} samples")
        print(f"✅ Test set: {test_count} samples")
        
        # Train model
        print("\nTraining Random Forest (100 trees, max_depth=15)...")
        model, train_metrics = classifier.train_random_forest(train)
        
        print("✅ Model trained successfully!")
        print(f"   Training Metrics:")
        print(f"     - AUC-ROC: {train_metrics['auc_roc']:.4f}")
        print(f"     - F1-Score: {train_metrics['f1_score']:.4f}")
        print(f"     - Samples: {train_metrics['training_samples']}")
        
        return True, model, test, train_metrics
    except Exception as e:
        print(f"❌ FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False, None, None, None


def test_evaluate_model(classifier, model, test_set):
    """Test 5: Can we evaluate the model?"""
    print("\n" + "="*80)
    print("TEST 5: EVALUATION TEST")
    print("="*80)
    
    try:
        test_count = test_set.count()
        print(f"Evaluating on {test_count} test samples...")
        test_metrics = classifier.evaluate_model(test_set, model)
        
        print("✅ Model evaluated successfully!")
        print(f"   Test Metrics:")
        print(f"     - Precision: {test_metrics['precision']:.4f} (need ≥ 0.80)")
        print(f"     - Recall: {test_metrics['recall']:.4f} (need ≥ 0.75)")
        print(f"     - F1-Score: {test_metrics['f1_score']:.4f} (need ≥ 0.77)")
        print(f"     - AUC-ROC: {test_metrics['auc_roc']:.4f}")
        
        # Check H1 criteria
        h1_met = (
            test_metrics['precision'] >= 0.80 and
            test_metrics['recall'] >= 0.75 and
            test_metrics['f1_score'] >= 0.77
        )
        
        if h1_met:
            print(f"\n✅ H1 HYPOTHESIS CRITERIA MET!")
        else:
            print(f"\n⚠️  H1 criteria not met (This is Phase 4 optimization goal)")
            print(f"   Current: P={test_metrics['precision']:.4f}, R={test_metrics['recall']:.4f}, F1={test_metrics['f1_score']:.4f}")
            print(f"   Target:  P≥0.80, R≥0.75, F1≥0.77")
        
        return True, test_metrics
    except Exception as e:
        print(f"❌ FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False, None


def test_feature_importance(classifier, model):
    """Test 6: Can we extract feature importance?"""
    print("\n" + "="*80)
    print("TEST 6: FEATURE IMPORTANCE TEST")
    print("="*80)
    
    try:
        print("Extracting top 10 features...")
        importance = classifier.feature_importance(model)
        
        if importance:
            print("✅ Feature importance extracted!")
            print(f"   Top 10 Features:")
            for i, (feature, score) in enumerate(importance.items(), 1):
                print(f"     {i:2d}. {feature:30s}: {score:.4f}")
        else:
            print("⚠️  No feature importance extracted (non-critical)")
        
        return True, importance
    except Exception as e:
        print(f"❌ FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False, None


def test_cross_validation(classifier, df_labeled):
    """Test 7: Can we perform cross-validation?"""
    print("\n" + "="*80)
    print("TEST 7: CROSS-VALIDATION TEST (5-fold)")
    print("="*80)
    
    try:
        print("Performing 5-fold cross-validation...")
        print("(This may take a few minutes...)")
        mean_metrics, std_metrics = classifier.cross_validate(df_labeled, num_folds=5)
        
        print("✅ Cross-validation completed!")
        print(f"   Mean Metrics:")
        for k, v in mean_metrics.items():
            print(f"     - {k}: {v:.4f}")
        
        return True, mean_metrics
    except Exception as e:
        print(f"❌ FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False, None


def main():
    """Run all tests"""
    print("\n" + "╔" + "="*78 + "╗")
    print("║" + " "*15 + "TESTING ml_classifier.py (PRODUCTION VERSION)" + " "*19 + "║")
    print("╚" + "="*78 + "╝")
    
    # Test 1: Import
    success = test_basic_import()
    if not success:
        print("\n❌ Cannot proceed: Import failed")
        return False
    
    # Test 2: Initialization
    success, pipeline, classifier = test_initialization()
    if not success:
        print("\n❌ Cannot proceed: Initialization failed")
        return False
    
    # Test 3: Data Loading
    success, df_labeled = test_load_data(pipeline, classifier)
    if not success:
        print("\n❌ Cannot proceed: Data loading failed")
        return False
    
    # Test 4: Training
    success, model, test_set, train_metrics = test_train_random_forest(classifier, df_labeled)
    if not success:
        print("\n❌ Cannot proceed: Training failed")
        return False
    
    # Test 5: Evaluation
    success, test_metrics = test_evaluate_model(classifier, model, test_set)
    if not success:
        print("\n❌ Cannot proceed: Evaluation failed")
        return False
    
    # Test 6: Feature Importance (optional)
    success, importance = test_feature_importance(classifier, model)
    if not success:
        print("\n⚠️  Feature importance test failed (non-critical)")
    
    # Test 7: Cross-validation (optional)
    success, cv_metrics = test_cross_validation(classifier, df_labeled)
    if not success:
        print("\n⚠️  Cross-validation test failed (non-critical, can retry later)")
    
    # Final summary
    print("\n" + "="*80)
    print("✅ TEST SUMMARY - ALL CORE TESTS PASSED")
    print("="*80)
    print("✅ Test 1: Import")
    print("✅ Test 2: Initialization")
    print("✅ Test 3: Data Loading")
    print("✅ Test 4: Training")
    print("✅ Test 5: Evaluation")
    
    if test_metrics:
        print("\n" + "="*80)
        print("FINAL METRICS SUMMARY")
        print("="*80)
        print(f"Precision:  {test_metrics['precision']:.4f}")
        print(f"Recall:     {test_metrics['recall']:.4f}")
        print(f"F1-Score:   {test_metrics['f1_score']:.4f}")
        print(f"AUC-ROC:    {test_metrics['auc_roc']:.4f}")
        
        h1_met = (
            test_metrics['precision'] >= 0.80 and
            test_metrics['recall'] >= 0.75 and
            test_metrics['f1_score'] >= 0.77
        )
        
        if h1_met:
            print("\n🎉 H1 HYPOTHESIS CRITERIA MET!")
            print("   Your classifier meets specification requirements!")
        else:
            print("\n⚠️  H1 criteria not fully met")
            print("   This is expected for initial testing")
            print("   Phase 4 will focus on optimizing these metrics")
    
    print("\n" + "="*80)
    print("✅ Your ml_classifier.py is working correctly!")
    print("="*80)
    
    return True


if __name__ == "__main__":
    success = main()
    exit(0 if success else 1)


╔==============================================================================╗
║               TESTING ml_classifier.py (PRODUCTION VERSION)                   ║
╚==============================================================================╝

TEST 1: IMPORT TEST
✅ SUCCESS: QualityClassifier imported successfully!

TEST 2: INITIALIZATION TEST


2026-01-02 18:11:08,369 - DataLoader - INFO - Creating Spark session...
INFO:DataLoader:Creating Spark session...
2026-01-02 18:11:08,925 - DataLoader - INFO - Spark version: 3.5.0
INFO:DataLoader:Spark version: 3.5.0
2026-01-02 18:11:08,926 - DataLoader - INFO - Master: local
INFO:DataLoader:Master: local
2026-01-02 18:11:08,927 - DataLoader - INFO - DataLoader initialized
INFO:DataLoader:DataLoader initialized
INFO:QualityAssessmentPipeline:QualityAssessmentPipeline initialized
2026-01-02 18:11:08,942 - DataLoader - INFO - Loading file: data/raw/titanic/train.csv
INFO:DataLoader:Loading file: data/raw/titanic/train.csv
2026-01-02 18:11:08,944 - CSVParser - INFO - Parsing CSV: data/raw/titanic/train.csv
INFO:CSVParser:Parsing CSV: data/raw/titanic/train.csv


✅ SUCCESS: QualityClassifier initialized!

TEST 3: DATA LOADING TEST
Loading Titanic dataset...


2026-01-02 18:11:11,912 - CSVParser - INFO - Loaded 891 rows from data/raw/titanic/train.csv
INFO:CSVParser:Loaded 891 rows from data/raw/titanic/train.csv
INFO:FeatureEngineer:Engineering all features...
INFO:quality_assessment.feature_engineer:Extracting row-level features...
INFO:quality_assessment.feature_engineer:Row-level features extracted
INFO:quality_assessment.feature_engineer:Extracting column-level features...


✅ Loaded 891 rows
Calculating column count...
   Total columns in raw data: 12
Extracting features...


...

✅ Model trained successfully!
   Training Metrics:
     - AUC-ROC: 1.0000
     - F1-Score: 1.0000
     - Samples: 746

TEST 5: EVALUATION TEST
Evaluating on 145 test samples...


INFO:quality_assessment.ml_classifier:Test Metrics:
INFO:quality_assessment.ml_classifier:  Precision: 1.0000 (need ≥ 0.80)
INFO:quality_assessment.ml_classifier:  Recall: 1.0000 (need ≥ 0.75)
INFO:quality_assessment.ml_classifier:  F1-Score: 1.0000 (need ≥ 0.77)
INFO:quality_assessment.ml_classifier:  AUC-ROC: 1.0000
INFO:quality_assessment.ml_classifier:Spec H1 Criteria Met: True
INFO:quality_assessment.ml_classifier:Top 10 Feature Importances:
INFO:quality_assessment.ml_classifier:  1. null_count: 0.3055
INFO:quality_assessment.ml_classifier:  2. null_ratio: 0.2840
INFO:quality_assessment.ml_classifier:  3. valid_types_count: 0.2836
INFO:quality_assessment.ml_classifier:  4. Age: 0.1140
INFO:quality_assessment.ml_classifier:  5. Fare: 0.0040
INFO:quality_assessment.ml_classifier:  6. SibSp: 0.0029
INFO:quality_assessment.ml_classifier:  7. Pclass: 0.0026
INFO:quality_assessment.ml_classifier:  8. Ticket: 0.0019
INFO:quality_assessment.ml_classifier:  9. Survived: 0.0008
INFO:quality

✅ Model evaluated successfully!
   Test Metrics:
     - Precision: 1.0000 (need ≥ 0.80)
     - Recall: 1.0000 (need ≥ 0.75)
     - F1-Score: 1.0000 (need ≥ 0.77)
     - AUC-ROC: 1.0000

✅ H1 HYPOTHESIS CRITERIA MET!

TEST 6: FEATURE IMPORTANCE TEST
Extracting top 10 features...
✅ Feature importance extracted!
   Top 10 Features:
      1. null_count                    : 0.3055
      2. null_ratio                    : 0.2840
      3. valid_types_count             : 0.2836
      4. Age                           : 0.1140
      5. Fare                          : 0.0040
      6. SibSp                         : 0.0029
      7. Pclass                        : 0.0026
      8. Ticket                        : 0.0019
      9. Survived                      : 0.0008
     10. Parch                         : 0.0005

TEST 7: CROSS-VALIDATION TEST (5-fold)
Performing 5-fold cross-validation...
(This may take a few minutes...)


INFO:quality_assessment.ml_classifier:  Training samples: 713
INFO:quality_assessment.ml_classifier:  Testing samples: 178
INFO:quality_assessment.ml_classifier:Preparing training data with target column: ground_truth
INFO:quality_assessment.ml_classifier:Preparing 16 features for vectorization
INFO:quality_assessment.ml_classifier:Type casting complete: 16 columns → DoubleType
INFO:quality_assessment.ml_classifier:VectorAssembly complete: 713 rows vectorized
INFO:quality_assessment.ml_classifier:Feature preparation complete: 16 features, 713 rows
INFO:quality_assessment.ml_classifier:Training Random Forest: 100 trees, max_depth=15
INFO:quality_assessment.ml_classifier:Model training complete
INFO:quality_assessment.ml_classifier:Training Metrics: AUC=1.0000, F1=1.0000, Samples=713
INFO:quality_assessment.ml_classifier:Evaluating model on test set (178 samples)
INFO:quality_assessment.ml_classifier:Test Metrics:
INFO:quality_assessment.ml_classifier:  Precision: 1.0000 (need ≥ 0.80)
IN

✅ Cross-validation completed!
   Mean Metrics:
     - f1_score: 1.0000
     - precision: 1.0000
     - recall: 1.0000
     - auc_roc: 1.0000

✅ TEST SUMMARY - ALL CORE TESTS PASSED
✅ Test 1: Import
✅ Test 2: Initialization
✅ Test 3: Data Loading
✅ Test 4: Training
✅ Test 5: Evaluation

FINAL METRICS SUMMARY
Precision:  1.0000
Recall:     1.0000
F1-Score:   1.0000
AUC-ROC:    1.0000

🎉 H1 HYPOTHESIS CRITERIA MET!
   Your classifier meets specification requirements!

✅ Your ml_classifier.py is working correctly!
